In [14]:
import re
from tqdm.notebook import tqdm
import multiprocessing as mp
from functools import partial
import os
import numpy as np
import itertools

import time
import humanize
import psutil

In [15]:
def load_keywords(keywords_file):
    """
    Load keywords from a file or use a predefined list
    """
    with open(keywords_file, 'r', encoding='utf-8') as f:

        # Read keywords one per line and strip whitespace
        keywords = []
        for line in f:
            line=line.strip()
            keywords.append(line)
    print(f"Loaded {len(keywords)} keywords from {keywords_file}")
    print(f"Using {len(keywords)} predefined keywords")
    
    return keywords

  # Load keywords

    

In [16]:
keywords_pattern = None

def check_tweet_bytes(tweet_bytes):
    if keywords_pattern.search(tweet_bytes):
        return tweet_bytes
    return None

def search_tweets_raw(input_file, output_file, keywords_file, chunk_size=10000000, mp_chunksize=50000):
    total_processed = 0
    total_matches = 0
    
    keywords = load_keywords(keywords_file)
    # Compile the regex pattern for bytes
    global keywords_pattern
    bytes_keywords = [kw.lower().encode('utf-8') for kw in keywords]
    pattern_string = b'|'.join(b'\\b' + re.escape(kw) + b'\\b' for kw in bytes_keywords)
    keywords_pattern = re.compile(pattern_string, re.IGNORECASE)
    
    with open(output_file, 'wb') as out_file:
        with mp.Pool(processes=mp.cpu_count()) as pool:
            with open(input_file, 'rb') as in_file:
                pbar = tqdm(desc="Processing file", unit="lines")
                
                while True:
                    # Read a chunk of lines
                    chunk = list(itertools.islice(in_file, chunk_size))
                    if not chunk:
                        break
                    
                    chunk_len = len(chunk)
                    total_processed += chunk_len
                    
                    # Process the chunk with multiprocessing
                    chunk_results = pool.imap(check_tweet_bytes, chunk, chunksize=mp_chunksize)
                    
                    # Track progress with tqdm for this chunk
                    chunk_matches = 0
                    for result in tqdm(chunk_results, total=chunk_len, desc=f"Chunk {total_processed//chunk_size}", leave=False):
                        if result is not None:
                            out_file.write(result)
                            chunk_matches += 1
                    
                    total_matches += chunk_matches
                    pbar.update(chunk_len)
                    pbar.set_postfix(matches=total_matches)
                
                pbar.close()
    
    print(f"Completed: {total_processed} lines processed, {total_matches} matches found ({total_matches/total_processed:.2%})")


In [19]:
 # Example usage/USA
input_file = '../USA/USA_March_June2020.txt'
output_file = 'EVS_Tweets_USA.txt'
keywords_file = 'Vicious_Keyword.txt'  

import os
print("Current working directory:", os.getcwd())

# Run search with keywords from file
start_time = time.time()
search_tweets_raw(input_file, output_file, keywords_file)
print("Time taken: {}".format(humanize.precisedelta(time.time() - start_time, suppress=['days', 'milliseconds', 'microseconds'])))
print("Memory used: {}".format(humanize.naturalsize(psutil.Process().memory_info().rss)))


Current working directory: /res_vol/Notebooks/BERT
Loaded 60 keywords from Vicious_Keyword.txt
Using 60 predefined keywords


Processing file: 0lines [00:00, ?lines/s]

Chunk 1:   0%|          | 0/10000000 [00:00<?, ?it/s]

Chunk 2:   0%|          | 0/10000000 [00:00<?, ?it/s]

Chunk 3:   0%|          | 0/10000000 [00:00<?, ?it/s]

Chunk 3:   0%|          | 0/4467587 [00:00<?, ?it/s]

Completed: 34467587 lines processed, 253226 matches found (0.73%)
Time taken: 3 minutes and 10 seconds
Memory used: 35.2 GB
